<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
# Desafio 4, LSTM Bot QA

### Datos
El objetivo es utilizar datos disponibles del challenge ConvAI2 (Conversational Intelligence Challenge 2) de conversaciones en inglés. Se construirá un BOT para responder a preguntas del usuario (QA).\
[LINK](http://convai.io/data/)

In [31]:
!pip install --upgrade --no-cache-dir gdown --quiet

Acceso denegado.


In [32]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf

# IMPORTACIONES CORRECTAS PARA TENSORFLOW 2.X (ANACONDA)
from tensorflow.keras.preprocessing.text import Tokenizer, one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Activation, Dropout, Dense, Flatten, LSTM, SimpleRNN, Embedding, Input
from sklearn.model_selection import train_test_split

In [33]:
# Descargar la carpeta de dataset
import os
import gdown
if os.access('data_volunteers.json', os.F_OK) is False:
    url = 'https://drive.google.com/uc?id=1awUxYwImF84MIT5-jCaYAPe2QwSgS1hN&export=download'
    output = 'data_volunteers.json'
    gdown.download(url, output, quiet=False)
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [34]:
# dataset_file
import json

text_file = "data_volunteers.json"
with open(text_file) as f:
    data = json.load(f) # la variable data será un diccionario



In [35]:
# Observar los campos disponibles en cada linea del dataset
data[0].keys()

dict_keys(['dialog', 'start_time', 'end_time', 'bot_profile', 'user_profile', 'eval_score', 'profile_match', 'participant1_id', 'participant2_id'])

In [36]:
chat_in = []
chat_out = []

input_sentences = []
output_sentences = []
output_sentences_inputs = []
max_len = 80 # Subimos el límite para tener más datos

def clean_text(txt):
    txt = txt.lower()
    # Guardamos los cambios asignando a la variable
    txt = txt.replace("\'d", " had")
    txt = txt.replace("\'s", " is")
    txt = txt.replace("\'m", " am")
    txt = txt.replace("don't", " do not")
    # Quitamos caracteres especiales pero mantenemos espacios
    txt = re.sub(r'[^a-z0-0\s]', '', txt)
    return txt.strip()
    
for line in data:
    for i in range(len(line['dialog'])-1):
        chat_in = clean_text(line['dialog'][i]['text'])
        chat_out = clean_text(line['dialog'][i+1]['text'])

        # Filtramos solo si son extremadamente largas o vacías
        if len(chat_in) < 1 or len(chat_out) < 1 or len(chat_in) > max_len:
            continue

        input_sentences.append(chat_in)
        output_sentences.append(chat_out + ' <eos>')
        output_sentences_inputs.append('<sos> ' + chat_out)

print("Cantidad de rows utilizadas ahora:", len(input_sentences))

Cantidad de rows utilizadas: 6033


In [37]:
input_sentences[1], output_sentences[1], output_sentences_inputs[1]

('hi how are you ', 'not bad and you  <eos>', '<sos> not bad and you ')

### ***Pueden realizar el desafio en Keras o PyTorch.***

### 2 - Preprocesamiento
Realizar el preprocesamiento necesario para obtener:
- word2idx_inputs, max_input_len
- word2idx_outputs, max_out_len, num_words_output
- encoder_input_sequences, decoder_output_sequences, decoder_targets

In [38]:
# Re-definimos los tokens especiales
SOS = '<sos>'
EOS = '<eos>'

input_sentences = []
output_sentences = [] # El target (lo que queremos que prediga)
output_sentences_inputs = [] # Lo que entra al decoder (teacher forcing)

for line in data:
    for i in range(len(line['dialog'])-1):
        chat_in = clean_text(line['dialog'][i]['text'])
        chat_out = clean_text(line['dialog'][i+1]['text'])

        if len(chat_in) < 2 or len(chat_out) < 2: # Filtro mínimo
            continue

        input_sentences.append(chat_in)
        output_sentences.append(chat_out + ' ' + EOS)
        output_sentences_inputs.append(SOS + ' ' + chat_out)

# Tokenización de entradas
tokenizer_in = Tokenizer(filters='')
tokenizer_in.fit_on_texts(input_sentences)
encoder_input_sequences = tokenizer_in.texts_to_sequences(input_sentences)
word2idx_inputs = tokenizer_in.word_index
max_input_len = max(len(s) for s in encoder_input_sequences)

# Tokenización de salidas
tokenizer_out = Tokenizer(filters='')
tokenizer_out.fit_on_texts(output_sentences + output_sentences_inputs)
decoder_input_sequences = tokenizer_out.texts_to_sequences(output_sentences_inputs)
decoder_target_sequences = tokenizer_out.texts_to_sequences(output_sentences)
word2idx_outputs = tokenizer_out.word_index
max_out_len = max(len(s) for s in decoder_target_sequences)
num_words_output = len(word2idx_outputs) + 1

# Padding
encoder_input_sequences = pad_sequences(encoder_input_sequences, maxlen=max_input_len, padding='post')
decoder_input_sequences = pad_sequences(decoder_input_sequences, maxlen=max_out_len, padding='post')
decoder_target_sequences = pad_sequences(decoder_target_sequences, maxlen=max_out_len, padding='post')

### 3 - Preparar los embeddings
Utilizar los embeddings de Glove o FastText para transformar los tokens de entrada en vectores

In [39]:
# Descarga de Glove (si no lo tenés en el entorno)
# !wget https://nlp.stanford.edu/data/glove.6B.zip
# !unzip glove.6B.zip

embeddings_index = {}
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

# Crear la matriz para el Encoder
num_words_in = len(word2idx_inputs) + 1
embedding_matrix_in = np.zeros((num_words_in, 100))
for word, i in word2idx_inputs.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix_in[i] = embedding_vector

### 4 - Entrenar el modelo
Entrenar un modelo basado en el esquema encoder-decoder utilizando los datos generados en los puntos anteriores. Utilce como referencias los ejemplos vistos en clase.

In [40]:
from keras.layers import Input, LSTM, Embedding, Dense
from keras.models import Model

latent_dim = 256

# ENCODER
encoder_inputs = Input(shape=(max_input_len,))
enc_emb = Embedding(num_words_in, 100, weights=[embedding_matrix_in], trainable=False)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c] # Estos estados son el "contexto"

# DECODER
decoder_inputs = Input(shape=(max_out_len,))
# Para el decoder solemos usar un embedding que se entrene de cero (o Glove también)
dec_emb_layer = Embedding(num_words_output, 100) 
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(num_words_output, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Modelo para entrenamiento
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Nota: decoder_target_sequences debe tener forma (N, max_out_len, 1) o usar sparse_loss
model.fit([encoder_input_sequences, decoder_input_sequences], decoder_target_sequences, batch_size=64, epochs=50)

Epoch 1/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 205s 981ms/step - accuracy: 0.9713 - loss: 0.4728
Epoch 2/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 247s 1s/step - accuracy: 0.9786 - loss: 0.1270
Epoch 3/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 258s 1s/step - accuracy: 0.9794 - loss: 0.1203
Epoch 4/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 257s 1s/step - accuracy: 0.9807 - loss: 0.1120
Epoch 5/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 260s 1s/step - accuracy: 0.9819 - loss: 0.1034
Epoch 6/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 259s 1s/step - accuracy: 0.9829 - loss: 0.0968
Epoch 7/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 259s 1s/step - accuracy: 0.9836 - loss: 0.0921
Epoch 8/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 260s 1s/step - accuracy: 0.9840 - loss: 0.0882
Epoch 9/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 259s 1s/step - accuracy: 0.9844 - loss: 0.0851
Epoch 10/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 259s 1s/step - accuracy: 0.9847 - loss: 0.0825
Epoch 11/50
208/208 ━━━━━━━━━━━━━━━━━━━━ 259s 1s/step - accuracy: 0.9850 - loss: 0.0801
Epoch 12/50
208/208 ━━━━━━━━━━━━━━━━━━

### 5 - Inferencia
Experimentar el funcionamiento de su modelo. Recuerde que debe realizar la inferencia de los modelos por separado de encoder y decoder.

In [42]:
idx2word_inputs = {v: k for k, v in word2idx_inputs.items()}
idx2word_outputs = {v: k for k, v in word2idx_outputs.items()}

# 1. Modelo del Encoder
encoder_model = Model(encoder_inputs, encoder_states)

# 2. Modelo del Decoder (Inferencia)
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb_inf = dec_emb_layer(decoder_inputs)
decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(dec_emb_inf, initial_state=decoder_states_inputs)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs_inf] + decoder_states_inf)

def decode_sequence(input_seq):
    # Obtener estados del encoder
    states_value = encoder_model.predict(input_seq)
    
    # Generar secuencia de target vacía con el token SOS
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = word2idx_outputs['<sos>'] # Necesitás haber definido este token en el preproceso

    stop_condition = False
    decoded_sentence = ''
    
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        
        # Elegir la palabra con mayor probabilidad
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = idx2word_outputs[sampled_token_index] # Diccionario inverso
        
        if sampled_word == '<eos>' or len(decoded_sentence.split()) > max_out_len:
            stop_condition = True
        else:
            decoded_sentence += ' ' + sampled_word
            
        # Actualizar la secuencia target y los estados
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]
        
    return decoded_sentence

In [43]:
def chat_with_bot(text):
    # Limpiamos y tokenizamos la entrada del usuario
    text_cleaned = clean_text(text)
    input_seq = tokenizer_in.texts_to_sequences([text_cleaned])
    input_seq = pad_sequences(input_seq, maxlen=max_input_len, padding='post')
    
    # Generamos la respuesta
    response = decode_sequence(input_seq)
    
    print(f"User: {text}")
    print(f"Bot: {response.strip()}")
    print("-" * 30)

# --- E. PRUEBAS FINALES ---
# Ahora podés hablarle al bot:
chat_with_bot("hello")
chat_with_bot("how are you")
chat_with_bot("do you have any pets")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
User: hello
Bot: i am a teacher
------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
User: how are you
Bot: i am a teacher
------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
User: do you have any pets
Bot: i am a teacher
------------------------------
